# 🕵️‍♂️ Análise Exploratória - Setor Financeiro B3

Este caderno unifica a coleta, o tratamento e a visualização de dados do mercado financeiro. Nosso objetivo é transformar números frios (Preços, Selic, IPCA, Lucros) em inteligência de investimento acionável.

OBS: Antes de importar as bibliotecas tem certeza que os datasets foram gerados pelos arquivos python responsáveis pela extração

## Importação das bibliotecas responsáveis pelos calculos e geração dos gráficos

In [ ]:
import os
import pandas as pd
from IPython.display import display
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
import warnings

In [16]:

#Verificação dos datasets brutos sem tratamento
notebook_dir = os.getcwd()
raw_data_path = os.path.join(notebook_dir, '..', 'data', 'raw')
raw_data_path = os.path.abspath(raw_data_path)

print(f"Procurando dados em: {raw_data_path}")
print(f"Diretório existe? {os.path.exists(raw_data_path)}")

if os.path.exists(raw_data_path):
    print("\nArquivos disponíveis no diretório './data/raw':")
    for filename in os.listdir(raw_data_path):
        print(f"- {filename}")
else:
    print(f"\n⚠️ Diretório não encontrado. Verifique se os dados foram extraídos.")
    print(f"Caminho esperado: {raw_data_path}")

Procurando dados em: c:\Users\breno\Desktop\Alpha EdTech\HardSkills\Python\Projeto\Desafio-Analytics\data\raw
Diretório existe? True

Arquivos disponíveis no diretório './data/raw':
- 01_yfinance_precos_raw.csv
- 02_yfinance_eventos_raw.csv
- 03_yfinance_info_raw.csv
- 04_yfinance_balancos_raw.csv
- 05_CVM_Historico_Focado.csv
- indicadores_economicos_10anos.csv


### Dataframes para demonstrar as colunas e as primeiras e ultimas linhas das tabelas brutas (sem tratamento de dados)

In [17]:

def ler_csv_robusto(caminho_arquivo: str):
    tentativas = [
        {"sep": ";", "encoding": "utf-8-sig", "engine": "python"},
        {"sep": ";", "encoding": "latin1", "engine": "python"},
        {"sep": ",", "encoding": "utf-8-sig", "engine": "python"},
        {"sep": ",", "encoding": "latin1", "engine": "python"},
        {"sep": None, "encoding": "utf-8-sig", "engine": "python"},  # autodetect
    ]

    ultimo_erro = None
    for cfg in tentativas:
        try:
            df = pd.read_csv(
                caminho_arquivo,
                sep=cfg["sep"],
                encoding=cfg["encoding"],
                engine=cfg["engine"],
                on_bad_lines="warn",
                dtype=str
            )
            return df, cfg
        except Exception as e:
            ultimo_erro = e

    raise ultimo_erro

cwd = os.getcwd()
candidatos = [
    os.path.abspath(os.path.join(cwd, "..", "data", "raw")),
    os.path.abspath(os.path.join(cwd, "data", "raw")),
]

raw_data_path = next((p for p in candidatos if os.path.isdir(p)), None)

if raw_data_path is None:
    print("⚠️ Diretório 'data/raw' não encontrado.")
    print("Caminhos testados:")
    for p in candidatos:
        print(f"- {p}")
else:
    print(f"📂 Usando: {raw_data_path}")

    arquivos = [
        f for f in os.listdir(raw_data_path)
        if f.lower().endswith((".csv", ".xlsx", ".xls", ".parquet"))
    ]

    if not arquivos:
        print("⚠️ Nenhum arquivo tabular encontrado em data/raw.")
    else:
        for arquivo in sorted(arquivos):
            caminho_arquivo = os.path.join(raw_data_path, arquivo)
            print("\n" + "=" * 80)
            print(f"📄 Arquivo: {arquivo}")

            try:
                if arquivo.lower().endswith(".csv"):
                    df, cfg = ler_csv_robusto(caminho_arquivo)
                    print(f"Leitura CSV: sep={cfg['sep']} | encoding={cfg['encoding']}")
                elif arquivo.lower().endswith((".xlsx", ".xls")):
                    df = pd.read_excel(caminho_arquivo)
                elif arquivo.lower().endswith(".parquet"):
                    df = pd.read_parquet(caminho_arquivo)
                else:
                    continue

                print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas")
                print(f"Colunas: {list(df.columns)}")

                print("\n🔹 Primeiras 5 linhas:")
                display(df.head())

                print("🔹 Últimas 5 linhas:")
                display(df.tail())

            except Exception as e:
                print(f"❌ Erro ao ler '{arquivo}': {e}")

📂 Usando: c:\Users\breno\Desktop\Alpha EdTech\HardSkills\Python\Projeto\Desafio-Analytics\data\raw

📄 Arquivo: 01_yfinance_precos_raw.csv
Leitura CSV: sep=; | encoding=utf-8-sig
Shape: 979132 linhas x 8 colunas
Colunas: ['Date', 'Ticker', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']

🔹 Primeiras 5 linhas:


,Date,Ticker,Adj Close,Close,High,Low,Open,Volume
0,2016-03-09,ABCB4.SA,NaN,"4,434516906738281","4,786898419851679","4,416210738828448","4,622148548271284","398209,0"
1,2016-03-09,ABEV3.SA,NaN,"11,678589820861816","12,054308774795915","11,634756161872069","12,04804665288629","21459800,0"
2,2016-03-09,ADHM3.SA,NaN,"1,343027949333191","1,343027949333191","1,343027949333191","1,343027949333191","2062,0"
3,2016-03-09,AELP3.SA,NaN,"8,88532829284668","8,88532829284668","7,779598662719662","7,779598662719662","7700,0"
4,2016-03-09,AFLT3.SA,NaN,"1,2182115316390991","1,2182115316390991","1,2182115316390991","1,2182115316390991","0,0"


🔹 Últimas 5 linhas:


,Date,Ticker,Adj Close,Close,High,Low,Open,Volume
979127,2026-03-06,WHRL3.SA,NaN,"4,329999923706055","4,389999866485596","4,329999923706055","4,360000133514404","1300,0"
979128,2026-03-06,WIZC3.SA,NaN,"9,350000381469727","9,420000076293945","9,220000267028809","9,359999656677246","399100,0"
979129,2026-03-06,WLMM3.SA,NaN,"23,979999542236328","23,979999542236328","23,979999542236328","23,979999542236328","300,0"
979130,2026-03-06,WLMM4.SA,NaN,"23,469999313354492","23,469999313354492","22,760000228881836","22,760000228881836","400,0"
979131,2026-03-06,YDUQ3.SA,NaN,"11,869999885559082","12,329999923706055","11,800000190734863","12,010000228881836","3144000,0"



📄 Arquivo: 02_yfinance_eventos_raw.csv
Leitura CSV: sep=; | encoding=utf-8-sig
Shape: 11939 linhas x 5 colunas
Colunas: ['Date', 'Dividends', 'Stock Splits', 'Ticker', 'Capital Gains']

🔹 Primeiras 5 linhas:


...

,DENOM_CIA,CD_CVM,DT_FIM_EXERC,01_Receita_Liquida_R$,02_Lucro_Bruto_R$,03_EBIT_Operacional_R$,04_Resultado_Financeiro_R$,05_Lucro_Antes_Impostos_EBT_R$,06_Lucro_Liquido_R$,Ticker_Alvo,Ano_Exercicio
1782,ZAMP SA,24317,2020-12-31,"2238127000,0","1324170000,0","-387165000,0","-44251000,0","-445607000,0","-445607000,0",ZAMP3,2020
1783,ZAMP SA,24317,2021-12-31,"2753287000,0","1718682000,0","-172636000,0","-97962000,0","-273841000,0","-273841000,0",ZAMP3,2021
1784,ZAMP SA,24317,2022-12-31,"3644674000,0","2346889000,0","96744000,0","-143888000,0","-55786000,0","-55786000,0",ZAMP3,2022
1785,ZAMP SA,24317,2023-12-31,"3841961000,0","2556809000,0","99257000,0","-178883000,0","-97826000,0","-97826000,0",ZAMP3,2023
1786,ZAMP SA,24317,2024-12-31,"4556360000,0","2958462000,0","-35632000,0","-173088000,0","-191319000,0","-191319000,0",ZAMP3,2024



📄 Arquivo: indicadores_economicos_10anos.csv
Leitura CSV: sep=; | encoding=utf-8-sig
Shape: 2558 linhas x 4 colunas
Colunas: ['data', 'selic', 'ipca', 'dolar']

🔹 Primeiras 5 linhas:


,data,selic,ipca,dolar
0,2016-03-01,NaN,"0,43",NaN
1,2016-03-07,"0,052531",NaN,"3,7714"
2,2016-03-08,"0,052531",NaN,"3,7813"
3,2016-03-09,"0,052531",NaN,"3,7037"
4,2016-03-10,"0,052531",NaN,"3,67"


🔹 Últimas 5 linhas:


,data,selic,ipca,dolar
2553,2026-02-26,"0,055131",NaN,"5,1382"
2554,2026-02-27,"0,055131",NaN,"5,1495"
2555,2026-03-02,"0,055131",NaN,"5,2001"
2556,2026-03-03,"0,055131",NaN,"5,287"
2557,2026-03-04,"0,055131",NaN,"5,2091"


# Tratamento dos Dados

Os dados brutos(raw) foram tratados na execução dos arquivos python e estão em market_data_processed.csv

In [20]:
# Localiza o arquivo processado
cwd = os.getcwd()
candidatos = [
    os.path.abspath(os.path.join(cwd, "..", "data", "processed", "01_market_data_processed.csv")),
    os.path.abspath(os.path.join(cwd, "data", "processed", "01_market_data_processed.csv")),
]

processed_file = next((p for p in candidatos if os.path.isfile(p)), None)

if processed_file is None:
    print("⚠️ Arquivo '01_market_data_processed.csv' não encontrado.")
    print("Caminhos testados:")
    for p in candidatos:
        print(f"- {p}")
else:
    print(f"📂 Usando: {processed_file}\n")

    try:
        # Usa a mesma função robusta de leitura
        df, cfg = ler_csv_robusto(processed_file)
        print(f"Leitura CSV: sep={cfg['sep']} | encoding={cfg['encoding']}\n")

        print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas")
        print(f"\n📋 Todas as colunas:")
        for i, col in enumerate(df.columns, 1):
            print(f"  {i}. {col}")

        print("\n🔹 Primeiras 5 linhas:")
        display(df.head())

        print("\n🔹 Últimas 5 linhas:")
        display(df.tail())

    except Exception as e:
        print(f"❌ Erro ao ler arquivo: {e}")

📂 Usando: c:\Users\breno\Desktop\Alpha EdTech\HardSkills\Python\Projeto\Desafio-Analytics\data\processed\01_market_data_processed.csv

Leitura CSV: sep=; | encoding=utf-8-sig

Shape: 998883 linhas x 54 colunas

📋 Todas as colunas:
  1. date
  2. ticker
  3. close
  4. high
  5. low
  6. open
  7. volume
  8. selic
  9. ipca
  10. dolar
  11. sector
  12. industry
  13. marketcap
  14. cvm_book_value
  15. roe_static
  16. dividends
  17. stock splits
  18. cvm_ebit
  19. cvm_lucro_liquido
  20. cvm_patrimonio_liquido
  21. retorno_diario
  22. log_return
  23. retorno_acumulado
  24. volatilidade_21d
  25. volatilidade_63d
  26. volatilidade_252d
  27. sma_21
  28. sma_50
  29. sma_200
  30. spread_sma50
  31. tendencia_alta_50_200
  32. momentum_1m
  33. momentum_3m
  34. momentum_6m
  35. momentum_12m
  36. momentum_anomalia_12m_1m
  37. obv
  38. vol_medio_20d
  39. volume_relativo
  40. rsi_14
  41. bb_upper
  42. bb_lower
  43. macd_line
  44. macd_signal
  45. macd_hist
  46. dra

,date,ticker,close,high,low,open,volume,selic,ipca,dolar,...,macd_hist,drawdown,roe,dy_diario,dy_ltm,p_l,p_vp,payout_ratio,tendencia_dy_3y,sharpe_ratio_21d
0,2016-10-28,AALR3.SA,"18,93486213684082","19,487128400537586","18,6587280644877","19,033479701743847","6342600,0","0,05166",NaN,"3,1815",...,"0,0","-5,281264314760392e-11","-1631781,8387161933","0,0","0,0","-60,77414576953365","99170147,33021557","-0,0",Lateralização,NaN
1,2016-10-31,AALR3.SA,"17,81060218811035","18,93486078141952","17,26819809593291","18,924998649433284","2523300,0","0,05166",NaN,"3,1811",...,"-0,07174764345174367","-0,05937513257536031","-1631781,8387161933","0,0","0,0","-60,77414576953365","99170147,33021557","-0,0",Lateralização,NaN
2,2016-11-01,AALR3.SA,"17,65281105041504","18,126182078569247","16,923030245091653","17,81060139313311","996200,0","0,05166","0,18","3,2053",...,"-0,12286288074372465","-0,06770849864618955","-1631781,8387161933","0,0","0,0","-60,77414576953365","99170147,33021557","-0,0",Lateralização,NaN
3,2016-11-03,AALR3.SA,"17,74156951904297","17,988117180865704","17,070959577923624","17,751431651237002","621000,0","0,05166","0,18","3,231",...,"-0,14235233111959075","-0,06302092986529262","-1631781,8387161933","0,0","0,0","-60,77414576953365","99170147,33021557","-0,0",Lateralização,NaN
4,2016-11-04,AALR3.SA,"17,5048828125","17,86977415510886","17,46543428637612","17,702121681100884","389800,0","0,05166","0,18","3,2449",...,"-0,1614423827788511","-0,07552097897153653","-1631781,8387161933","0,0","0,0","-60,77414576953365","99170147,33021557","-0,0",Lateralização,NaN



🔹 Últimas 5 linhas:


,date,ticker,close,high,low,open,volume,selic,ipca,dolar,...,macd_hist,drawdown,roe,dy_diario,dy_ltm,p_l,p_vp,payout_ratio,tendencia_dy_3y,sharpe_ratio_21d
998878,2026-02-27,YDUQ3.SA,"13,270000457763672","13,529999732971191","13,210000038146973","13,359999656677246","2312500,0","0,055131","0,33","5,1495",...,"-0,033341781105723","-0,7253198540529063","28680171,469388906","0,0","0,08899283420281213","9,338495228716795","267829644,42566785","0,8310591575929458",Crescimento,"-3,0488556498206183"
998879,2026-03-02,YDUQ3.SA,"13,010000228881836","13,239999771118164","12,859999656677246","13,09000015258789","2950700,0","0,055131","0,33","5,2001",...,"-0,06167203453107867","-0,7307016851269028","28680171,469388906","0,0","0,08899283420281213","9,338495228716795","267829644,42566785","0,8310591575929458",Crescimento,"-3,451927533691368"
998880,2026-03-03,YDUQ3.SA,"12,100000381469728","12,529999732971191","12,06999969482422","12,4399995803833","3765100,0","0,055131","0,33","5,287",...,"-0,13527482473487465","-0,7495380741454692","28680171,469388906","0,0","0,08899283420281213","9,338495228716795","267829644,42566785","0,8310591575929458",Crescimento,"-4,201805817249713"
998881,2026-03-04,YDUQ3.SA,"12,300000190734863","12,65999984741211","12,239999771118164","12,399999618530272","1986400,0","0,055131","0,33","5,2091",...,"-0,16224928041487605","-0,7453982116810187","28680171,469388906","0,0","0,08899283420281213","9,338495228716795","267829644,42566785","0,8310591575929458",Crescimento,"-4,470315994497615"
998882,2026-03-05,YDUQ3.SA,"12,09000015258789","12,399999618530272","11,859999656677246","12,199999809265137","4946700,0","0,055131","0,33","5,2091",...,"-0,18437938731411785","-0,7497450722037972","28680171,469388906","0,0","0,08899283420281213","9,338495228716795","267829644,42566785","0,8310591575929458",Crescimento,"-4,5897320552275875"


In [33]:
# Dicionário de metadados: explicação de cada coluna do dataset processado
colunas_explicacao = {
    'Date': 'Data da observação (formato YYYY-MM-DD)',
    'Ticker': 'Código do ativo na B3 (ex: ITUB4.SA)',
    'Open': 'Preço de abertura do pregão',
    'High': 'Preço máximo atingido no dia',
    'Low': 'Preço mínimo atingido no dia',
    'Close': 'Preço de fechamento do pregão',
    'Volume': 'Volume financeiro negociado (R$)',
    'Adj Close': 'Preço de fechamento ajustado por proventos',
    'Dividends': 'Valor de dividendos pagos na data',
    'Stock Splits': 'Desdobramento/Grupamento de ações (ex: 2.0 = 2 para 1)',
    'SELIC': 'Taxa Selic anualizada (% a.a.)',
    'IPCA': 'Índice de Preços ao Consumidor Amplo (% mensal)',
    'USD_BRL': 'Taxa de câmbio Dólar/Real (PTAX)',
    'Return': 'Retorno diário do ativo (%)',
    'Volatility': 'Volatilidade móvel (desvio padrão dos retornos)',
    'MA_20': 'Média móvel simples de 20 dias',
    'MA_50': 'Média móvel simples de 50 dias',
    'RSI': 'Índice de Força Relativa (0-100)',
    'MACD': 'Moving Average Convergence Divergence',
    'Signal_Line': 'Linha de sinal do MACD',
    'BB_Upper': 'Banda de Bollinger superior',
    'BB_Lower': 'Banda de Bollinger inferior'
}

# Exibir como DataFrame
df_metadados = pd.DataFrame([
    {'Coluna': col, 'Descrição': desc}
    for col, desc in colunas_explicacao.items()
])

print("📖 Dicionário de Metadados - Market Data Processed:\n")
display(df_metadados)

print(f"\n📊 Total de colunas documentadas: {len(df_metadados)}")

📖 Dicionário de Metadados - Market Data Processed:



,Coluna,Descrição
0,Date,Data da observação (formato YYYY-MM-DD)
1,Ticker,Código do ativo na B3 (ex: ITUB4.SA)
2,Open,Preço de abertura do pregão
3,High,Preço máximo atingido no dia
4,Low,Preço mínimo atingido no dia
5,Close,Preço de fechamento do pregão
6,Volume,Volume financeiro negociado (R$)
7,Adj Close,Preço de fechamento ajustado por proventos
8,Dividends,Valor de dividendos pagos na data
9,Stock Splits,Desdobramento/Grupamento de ações (ex: 2.0 = 2...



📊 Total de colunas documentadas: 22


### 🗂️ Universo de Análise (Os 32 Tickers Selecionados)
Focaremos estritamente nos ativos do setor financeiro:
`BBSE3, CXSE3, PSSA3, WIZC3, ITUB4, BPAC11, BBDC3, ITSA4, BBAS3, SANB11, B3SA3, MULT3, BPAN4, ALOS3, BRAP4, IGTI3, BRSR6, ABCB4, SIMH3, IRBR3, BEES3, BMGB4, PLPL3, PINE4, LOGG3, BGIP4, SCAR3, SYNE3, MERC4, RPAD3, ESPA3, HBRE3`

In [21]:
warnings.filterwarnings('ignore')

tickers_selecionados = [
    'BBSE3.SA', 'CXSE3.SA', 'PSSA3.SA', 'WIZC3.SA', 'ITUB4.SA', 'BPAC11.SA',
    'BBDC3.SA', 'ITSA4.SA', 'BBAS3.SA', 'SANB11.SA', 'B3SA3.SA', 'MULT3.SA',
    'BPAN4.SA', 'ALOS3.SA', 'BRAP4.SA', 'IGTI3.SA', 'BRSR6.SA', 'ABCB4.SA',
    'SIMH3.SA', 'IRBR3.SA', 'BEES3.SA', 'BMGB4.SA', 'PLPL3.SA', 'PINE4.SA',
    'LOGG3.SA', 'BGIP4.SA', 'SCAR3.SA', 'SYNE3.SA', 'MERC4.SA', 'RPAD3.SA',
    'ESPA3.SA', 'HBRE3.SA'
]
print("✅ Ambiente configurado. Bibliotecas importadas.")

✅ Ambiente configurado. Bibliotecas importadas.


In [32]:
nomes_bancos = [
    'Banco do Brasil Seguridade', 'Caixa Seguridade', 'Portoseg', 'Wizz Capital',
    'Itaú Unibanco', 'Banco Pan', 'Banco do Brasil', 'Itaúsa', 'Banco da Amazônia',
    'Banco Santander', 'B3 S.A.', 'Multibanco', 'Banco PAN', 'Alos',
    'Banco Bradesco', 'Banco Inovação Digital', 'Banco Investcorp', 'ABC Brasil',
    'Banco Simh', 'Banco IRB Brasil', 'Banco Beés', 'Banco BMG', 'Plural Bank',
    'Pine Investimentos', 'Loggi', 'BGI Participações', 'Scarpa',
    'Syne Consultoria', 'Mercado Bitcoin', 'Republica da Plataforma', 'Espara',
    'Habib Bank'
]

# Criar dicionário com nomes dos bancos
tickers_dict = {i+1: (nomes_bancos[i], tickers_selecionados[i]) 
                for i in range(len(tickers_selecionados))}

# Criar DataFrame com apenas Banco e Ticker
df_tickers = pd.DataFrame([
    {'Banco': banco, 'Ticker': ticker}
    for banco, ticker in zip(nomes_bancos, tickers_selecionados)
])

print("✅ Relação de Bancos e Tickers (Setor Financeiro B3):\n")
display(df_tickers)

print(f"\n📊 Total de tickers: {len(df_tickers)}")

✅ Relação de Bancos e Tickers (Setor Financeiro B3):



,Banco,Ticker
0,Banco do Brasil Seguridade,BBSE3.SA
1,Caixa Seguridade,CXSE3.SA
2,Portoseg,PSSA3.SA
3,Wizz Capital,WIZC3.SA
4,Itaú Unibanco,ITUB4.SA
5,Banco Pan,BPAC11.SA
6,Banco do Brasil,BBDC3.SA
7,Itaúsa,ITSA4.SA
8,Banco da Amazônia,BBAS3.SA
9,Banco Santander,SANB11.SA



📊 Total de tickers: 32


# 📈 O que move o preço das ações? 

Quais fatores históricos ajudam a explicar o desempenho das ações brasileiras nos últimos 10 anos?

## O que são os dados macroeconomicos?
    A macroeconomia analisa o comportamento da economia como um todo, focando em indicadores amplos que afetam o país ou o mundo.

    Exemplos de Dados Macro:
        - SELIC (taxa de juros média praticada nas operações compromissadas com títulos públicos federais com prazo de um dia útil.)

        - IPCA (Inflação oficial do Brasil. Mede a variação de preços e afeta o poder de compra da população.)

        - USD-BRL (Taxa de câmbio Dólar/Real. Impacta importações, exportações e a economia como um todo.)

## O que são os dados microeconomicos?
    A microeconomia estuda como indivíduos e empresas tomam decisões e como essas escolhas interagem nos mercados. 
    Exemplos de Dados Micro:

   ...

## Gráfico de Evolução dos Ativos nos Ultimos 10 anos